In [1]:
import pandas as pd
import numpy as np
import pyarrow
from src_RN import *
import itertools, gc

from sklearn.model_selection import  train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import StratifiedKFold

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras import optimizers

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam

# Configurações 
print(tf.config.list_physical_devices())
BATCH_SIZE = 64

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.2- Pré-Processando os Dados

In [3]:
df = pre_processor_rn(df, objetivo = '', n_samples = 50000)

## 1.3- Construção da Matriz X e Vetor y

In [4]:
X = df.drop([ 'FALTOU'], axis=1)
y = df['FALTOU']

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## 1.5 - Tratando os Dados

In [6]:
preprocessador = transformar(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_val   = preprocessador.transform(X_val).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32)

## 1.6 - Treinando a Rede Neural

In [7]:
max_neurons = num_max_neuronio(X_train, d = X_train.shape[1])
print("Número máximo de neurônios:", max_neurons)

Número máximo de neurônios: 19


In [8]:
param_grid = {
    'neurons':       [18],
    'learning_rate': [0.1, 0.01, 0.001, 0.0001],   
    'batch_size':    [32,64],             
    'epochs':        [100],
    'l2_reg':        [0.001, 0.01, 0.1],          
    'dropout':       [0.0, 0.2],
}

# Gera todas as combinações
keys, values = zip(*param_grid.items())
combinacoes = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Total de combinações: {len(combinacoes)}")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

resultados = []

for params in combinacoes:
    print(f"Testando: {params}")
    
    accs = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = create_model(
            input_dim=X_train.shape[1],
            neurons=params['neurons'],
            learning_rate=params['learning_rate'],
            l2_reg=params['l2_reg'],
            dropout=params['dropout']
        )

        early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val_fold, y_val_fold),
            epochs=params['epochs'],  
            batch_size=params['batch_size'],
            callbacks=[early_stop],
            verbose=1
        )

        # Avaliação no fold
        loss, acc = model.evaluate(X_val_fold, y_val_fold, verbose=0)
        accs.append(acc)

        # Libera memória (importante no Keras)
        del model
        gc.collect()

    mean_acc = np.mean(accs)

    resultados.append({
        'params': params,
        'mean_accuracy': mean_acc
    })

    print(f"Accuracy média: {mean_acc:.4f}")

Total de combinações: 48
Testando: {'neurons': 13, 'learning_rate': 0.1, 'batch_size': 32, 'epochs': 100, 'l2_reg': 0.001, 'dropout': 0.0}


C:\Users\Micael\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7304 - loss: 0.5873 - val_accuracy: 0.7364 - val_loss: 0.5425
Epoch 2/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5631 - val_accuracy: 0.7364 - val_loss: 0.6010
Epoch 3/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7370 - loss: 0.5536 - val_accuracy: 0.7364 - val_loss: 0.5533
Epoch 4/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5692 - val_accuracy: 0.7364 - val_loss: 0.5463
Epoch 5/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7359 - loss: 0.5581 - val_accuracy: 0.7364 - val_loss: 0.5660
Epoch 6/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5618 - val_accuracy: 0.7364 - val_loss: 0.5457
Epoch 7/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5671 - val_accuracy: 0.7364 - val_loss: 0.5826
Epoch 8/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7365 - loss: 0.5767 - val_accu

C:\Users\Micael\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7290 - loss: 0.6070 - val_accuracy: 0.7364 - val_loss: 0.5524
Epoch 2/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5733 - val_accuracy: 0.7364 - val_loss: 0.5839
Epoch 3/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5794 - val_accuracy: 0.7364 - val_loss: 0.5943
Epoch 4/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5805 - val_accuracy: 0.7364 - val_loss: 0.5784
Epoch 5/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5787 - val_accuracy: 0.7364 - val_loss: 0.5769
Epoch 6/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5791 - val_accuracy: 0.7364 - val_loss: 0.5793
Epoch 7/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5788 - val_accuracy: 0.7364 - val_loss: 0.5789
Epoch 8/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7365 - loss: 0.5781 - val_accuracy: 0.7364

...

C:\Users\Micael\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7359 - loss: 0.5875 - val_accuracy: 0.7506 - val_loss: 0.5337
Epoch 2/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7491 - loss: 0.5432 - val_accuracy: 0.7509 - val_loss: 0.5388
Epoch 3/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7491 - loss: 0.5362 - val_accuracy: 0.7502 - val_loss: 0.5357
Epoch 4/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7518 - loss: 0.5356 - val_accuracy: 0.7546 - val_loss: 0.5323
Epoch 5/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7527 - loss: 0.5322 - val_accuracy: 0.7550 - val_loss: 0.5285
Epoch 6/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7529 - loss: 0.5340 - val_accuracy: 0.7556 - val_loss: 0.5319
Epoch 7/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7480 - loss: 0.5332 - val_accuracy: 0.7436 - val_loss: 0.5394
Epoch 8/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7526 - loss: 0.5314 - val_accuracy: 0.7458

C:\Users\Micael\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7371 - loss: 0.5826 - val_accuracy: 0.7493 - val_loss: 0.5453
Epoch 2/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7452 - loss: 0.5490 - val_accuracy: 0.7487 - val_loss: 0.5380
Epoch 3/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7455 - loss: 0.5444 - val_accuracy: 0.7496 - val_loss: 0.5418
Epoch 4/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7436 - loss: 0.5449 - val_accuracy: 0.7496 - val_loss: 0.5338
Epoch 5/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7475 - loss: 0.5392 - val_accuracy: 0.7521 - val_loss: 0.5283
Epoch 6/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7439 - loss: 0.5418 - val_accuracy: 0.7531 - val_loss: 0.5292
Epoch 7/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7507 - loss: 0.5420 - val_accuracy: 0.7515 - val_loss: 0.5306
Epoch 8/100
 70/199 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7383 - loss: 0.5463

KeyboardInterrupt: 